# 完整配音流水线（GPU）

依次执行：TIGER-DnR 分离原片 → NVIDIA RE-USE 修复演员干声 → 排 Reaper 工程。
顶部菜单选 **运行时 → 更改运行时类型 → T4 GPU**（免费）。

注意：RE-USE 是 NVIDIA 非商用许可（NSCLv1），毕业论文/非商业用途没问题。

In [ ]:
import shutil, subprocess, sys
shutil.rmtree('/content/MyDubbingMixingLab', ignore_errors=True)
!git clone -q https://github.com/Fectxd/MyDubbingMixingLab.git
%cd /content/MyDubbingMixingLab
!pip install -q einops pyyaml soundfile reathon

import torch
print('python', sys.version.split()[0], '| torch', torch.__version__, '| cuda', torch.cuda.is_available())
if not torch.__version__.startswith('2.10.'):
    print('mamba-ssm 官方轮子最高支持 torch 2.10，正在降级 torch ...')
    !pip install -q torch==2.10.0 torchvision==0.25.0 torchaudio==2.10.0 --index-url https://download.pytorch.org/whl/cu128
    print('降级完成！请点菜单 运行时 → 重启运行时，然后运行下一个格子')
else:
    py = f'cp{sys.version_info.major}{sys.version_info.minor}'
    url = f'https://github.com/state-spaces/mamba/releases/download/v2.3.2.post1/mamba_ssm-2.3.2.post1%2Bcu12torch2.10cxx11abiTRUE-{py}-{py}-linux_x86_64.whl'
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', url])
    if r.returncode != 0:
        print('wheel 安装失败，请把输出贴给助手')
    else:
        import mamba_ssm
        print('mamba_ssm OK')

### 如果上一格提示"请点菜单 运行时 → 重启运行时"：
先点 **运行时 → 重启运行时**，再运行下面这个格子（否则直接跳过本格）：

In [ ]:
%cd /content/MyDubbingMixingLab
import subprocess, sys
py = f'cp{sys.version_info.major}{sys.version_info.minor}'
url = f'https://github.com/state-spaces/mamba/releases/download/v2.3.2.post1/mamba_ssm-2.3.2.post1%2Bcu12torch2.10cxx11abiTRUE-{py}-{py}-linux_x86_64.whl'
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', url])
import mamba_ssm
print('mamba_ssm OK')

In [ ]:
import os
from google.colab import files
os.makedirs('test', exist_ok=True)
print('请上传：原片（原片.mp4）+ 5 条演员干声 wav（可多选）')
up = files.upload()
for name, data in up.items():
    with open(os.path.join('test', name), 'wb') as f:
        f.write(data)
print('已保存：', list(up))

In [ ]:
!python separate.py --input test/原片.mp4 --device auto

In [ ]:
!python enhance.py --inputs test/*.wav --outdir work/enhanced

In [ ]:
!python assemble_rpp.py --actors test

In [ ]:
!zip -rq output.zip work
from google.colab import files
files.download('output.zip')
print('下载完成后，解压 output.zip 即可得到全部结果')